<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/8%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_GPT%EB%AA%A8%EB%8D%B8%EC%9B%90%EB%A6%AC%EC%99%80%ED%85%8D%EC%8A%A4%ED%8A%B8%EC%83%9D%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8주차 복습 과제: GPT 모델의 원리와 텍스트 생성

> **복습 키워드**
> 1. 텍스트 모델에 대한 이해
> 2. GPT 모델의 원리와 Transformer와의 접점
> 3. GPT 전 버전의 성능과 앞으로 배울 최신 GPT와의 비교점 고민


---
## 1. 텍스트 모델에 대한 이해

텍스트를 처리하는 AI 모델은 크게 세 가지 패러다임으로 발전해 왔습니다.

### 패러다임 1: 전통 ML (TF-IDF + 분류기)

사람이 직접 특성(feature)을 설계하는 방식입니다.
텍스트를 단어 빈도 벡터로 변환(TF-IDF)하고, 이를 ML 분류기(Naive Bayes, SVM 등)에 입력합니다.
단어 순서와 문맥을 무시하지만, 적은 데이터로도 빠르게 결과를 얻을 수 있다는 장점이 있습니다.

### 패러다임 2: 사전학습 + 파인튜닝 (딥러닝)

대규모 텍스트로 모델이 스스로 언어의 규칙을 학습합니다.
RNN/LSTM은 단어를 순서대로 읽어 문맥을 누적하고, Transformer는 Self-Attention으로 모든 단어 간 관계를 동시에 파악합니다.
사전학습된 모델(BERT, GPT)을 특정 데이터로 미세 조정(파인튜닝)하면 높은 성능을 달성합니다.

### 패러다임 3: 프롬프트 (GPT-3 이후)

모델이 충분히 커지면 별도의 학습 없이 **질문(프롬프트)만으로** 다양한 과제를 수행할 수 있습니다.
감성 분류, 요약, 번역, 코드 작성 등을 하나의 모델이 프롬프트에 따라 처리합니다.
이것이 바로 ChatGPT가 동작하는 방식입니다.

### 발전 경로 요약

```
BoW/TF-IDF  →  Word2Vec  →  RNN/LSTM  →  Transformer  →  GPT  →  ChatGPT
 (단어 빈도)   (단어 의미)   (순서 이해)   (문맥 이해)     (생성)    (대화)
```

### 방법론 선택 가이드

| 상황 | 추천 방법 | 이유 |
|------|----------|------|
| 데이터가 적고 빠르게 결과가 필요 | TF-IDF + ML | 간단하고 설명 가능 |
| 단어 순서가 중요한 분류 문제 | LSTM | 순서 반영, 중간 규모 |
| 최고 성능이 필요한 분류/이해 | BERT 파인튜닝 | 양방향 문맥 이해 |
| 텍스트 생성이 필요 | GPT 파인튜닝 | 자연스러운 문장 생성 |
| 학습 데이터가 없음 | 프롬프트 (GPT API) | Zero/Few-shot으로 해결 |


---
## 2. GPT 모델의 원리와 Transformer와의 접점

### 2-1. GPT = Transformer Decoder + 다음 단어 예측

GPT의 정식 이름은 **Generative Pre-trained Transformer**입니다.
이름 그대로 Transformer 구조를 기반으로 대규모 텍스트를 사전학습한 생성 모델입니다.

Transformer에는 Encoder와 Decoder 두 가지 구조가 있습니다.

| 구분 | Transformer Encoder | Transformer Decoder (GPT) |
|------|--------------------|--------------------------|
| Attention 방향 | **양방향** — 모든 단어를 동시에 참조 | **단방향** — 왼쪽(이전) 단어만 참조 |
| 대표 모델 | BERT | GPT, ChatGPT |
| 주요 용도 | 분류, 정보 추출, 이해 | 텍스트 생성, 대화 |
| 학습 방식 | 빈칸 채우기 (Masked LM) | 다음 단어 예측 |

**핵심 차이는 딱 하나: Causal Mask(인과 마스크)**입니다.

### 2-2. Causal Mask란?

Encoder의 Self-Attention은 모든 단어가 서로를 참조할 수 있습니다.
하지만 GPT에서는 텍스트를 **생성**해야 하므로, 아직 만들어지지 않은 미래 단어를 볼 수 없습니다.

Causal Mask는 **삼각형 마스크**를 씌워서 각 단어가 자기보다 왼쪽(과거) 단어만 참조하도록 제한합니다.

```
문장: "나는  영화를  재미있게  봤다"

       나는  영화를  재미있게  봤다
나는     O     X       X       X     ← 자기 자신만 참조
영화를   O     O       X       X     ← 나는 + 자신
재미있게 O     O       O       X     ← 나는 + 영화를 + 자신
봤다     O     O       O       O     ← 모두 참조 가능

O = 참조 가능, X = 마스킹 (가려짐)
```

코드에서는 `torch.triu()`로 상삼각 행렬을 만들어 마스크로 사용합니다.
이 한 줄의 차이가 Encoder(분류 모델)를 Decoder(생성 모델)로 바꿉니다.


### 2-3. GPT의 학습 방식: 다음 단어 예측

GPT의 사전학습은 매우 단순한 원리입니다. **"지금까지의 단어들을 보고 다음 단어를 맞춰라."**

```
학습 데이터: "나는 영화를 좋아한다"

  입력: [나는]              → 정답: 영화를
  입력: [나는, 영화를]       → 정답: 좋아한다
```

이 작업을 인터넷의 수십억 개 문장에 대해 반복하면, 모델은 자연스럽게 문법, 상식, 논리, 세계 지식을 가중치 속에 학습하게 됩니다.

### 2-4. GPT의 텍스트 생성 과정: 자기회귀(Autoregressive) 생성

학습이 끝난 GPT는 다음과 같이 텍스트를 생성합니다.

```
사용자 입력: "인공지능 기술이 발전하면"

Step 1: "인공지능 기술이 발전하면"           → 다음 단어 = "사회의"
Step 2: "인공지능 기술이 발전하면 사회의"      → 다음 단어 = "많은"
Step 3: "인공지능 기술이 발전하면 사회의 많은"  → 다음 단어 = "부분이"
Step 4: ... (반복)
Step N: 종료 토큰 또는 최대 길이 도달 → 생성 중단
```

한 단어를 생성할 때마다 그 단어를 입력에 추가하고, 다시 다음 단어를 예측합니다.
이렇게 자기가 만든 출력을 다시 입력으로 사용하는 방식을 **자기회귀(Autoregressive) 생성**이라 합니다.

### 2-5. Transformer와의 접점 정리

GPT는 Transformer 구조를 그대로 사용합니다. 공유하는 핵심 요소는 다음과 같습니다.

| 공유 요소 | 설명 |
|-----------|------|
| **Self-Attention** | 모든 단어 쌍의 관련도를 계산 (GPT는 Causal Mask 적용) |
| **Positional Encoding** | 단어 순서 정보를 벡터에 추가 |
| **Multi-Head Attention** | 여러 관점에서 동시에 문맥 파악 |
| **Feed-Forward Network** | 각 위치에서 비선형 변환 수행 |
| **Layer Normalization** | 학습 안정화를 위한 정규화 |

차이점은 오직 두 가지입니다.

| 차이점 | Encoder (분류) | GPT (생성) |
|--------|---------------|------------|
| 마스크 | 없음 (양방향) | Causal Mask (단방향) |
| 출력층 | sigmoid → 1개 값 (긍정/부정) | softmax → 어휘 크기 (다음 단어 확률 분포) |


---
## 3. Temperature — 생성의 창의성 조절

GPT가 다음 단어를 선택할 때, 어휘 전체에 대한 확률 분포가 계산됩니다.
**Temperature**는 이 확률 분포의 "뾰족함"을 조절하는 파라미터입니다.

```
예) 다음 단어 후보: 맑아요(0.4), 좋아요(0.3), 흐려요(0.2), 나빠요(0.1)

Temperature = 0.3 (낮음) → 확률 분포가 뾰족 → 거의 항상 "맑아요" 선택
Temperature = 0.7 (중간) → 적당한 다양성    → 보통 "맑아요" 또는 "좋아요"
Temperature = 1.2 (높음) → 확률 분포가 평평 → "흐려요", "나빠요"도 가끔 선택
```

| Temperature | 특징 | 적합한 용도 |
|-------------|------|------------|
| 0.1 ~ 0.3 | 매우 보수적, 반복적 | 정확한 답변이 필요한 경우 (번역, 분류) |
| 0.5 ~ 0.8 | 적당한 다양성 (추천) | 일반적인 대화, 글쓰기 |
| 1.0 ~ 1.5 | 창의적, 예측 불가 | 브레인스토밍, 창작 |

수학적으로는 softmax 함수의 입력(logit)을 temperature로 나누는 것입니다.
값이 작을수록 logit 차이가 극대화되어 확률이 한쪽에 집중되고,
값이 클수록 logit 차이가 줄어들어 확률이 고르게 분포됩니다.


---
## 4. 프롬프트 엔지니어링

GPT-3 이후 모델이 충분히 커지면서, 별도의 학습 없이 **프롬프트(질문) 설계만으로** 다양한 과제를 수행할 수 있게 되었습니다.

### 4-1. 프롬프트의 세 가지 방식

**Zero-shot**: 예시 없이 바로 질문하는 방식입니다.
```
다음 리뷰가 긍정인지 부정인지 분류하세요.
리뷰: 가격 대비 만족합니다 →
```

**Few-shot**: 몇 가지 예시를 보여주고 패턴을 따르게 하는 방식입니다.
```
리뷰: 품질 좋아요 → 긍정
리뷰: 불량이에요 → 부정
리뷰: 가격 대비 만족합니다 →
```

**System Prompt**: 모델에게 역할을 부여하여 답변 품질을 높이는 방식입니다.
```
당신은 상품 리뷰 감성 분석 전문가입니다.
리뷰를 읽고 긍정/부정/중립 중 하나로 분류하세요.
```

### 4-2. 좋은 프롬프트의 4가지 원칙

| 원칙 | 설명 | 예시 |
|------|------|------|
| **역할 부여** | 모델에게 전문가 역할을 지정 | "당신은 감성 분석 전문가입니다" |
| **명확한 지시** | 출력 형식과 기준을 구체적으로 명시 | "긍정/부정/중립 중 하나로 답하세요" |
| **예시 제공** | Few-shot으로 패턴을 보여줌 | "품질 좋아요 → 긍정" |
| **제약 조건** | 답변 범위를 한정 | "한 단어로만 답하세요" |

### 4-3. 프롬프트 설계 예시

**감성 분류 프롬프트:**
```
당신은 상품 리뷰 감성 분석 전문가입니다.
리뷰를 읽고 긍정/부정/중립 중 하나로 분류하세요.

예시:
리뷰: 품질 좋고 배송 빨라요 → 긍정
리뷰: 불량이고 환불 어렵다 → 부정
리뷰: 디자인은 좋은데 내구성이 약해요 → 중립

리뷰: 가격 대비 만족합니다 →
```

**텍스트 요약 프롬프트:**
```
다음 텍스트를 3줄로 요약하세요.
각 줄은 핵심 내용만 담아주세요.

텍스트: (여기에 긴 텍스트)

요약:
```

**키워드 추출 프롬프트:**
```
다음 텍스트에서 핵심 키워드 5개를 추출하세요.
쉼표로 구분하여 나열하세요.

텍스트: (여기에 텍스트)

키워드:
```

이런 프롬프트는 `skt/kogpt2` 같은 소규모 모델에서는 잘 동작하지 않지만,
GPT-4, Claude 같은 대형 모델에서는 정확한 결과를 반환합니다.
모델의 크기가 커질수록 프롬프트만으로 수행 가능한 과제의 범위가 넓어집니다.


---
## 5. 코드 실행 — 미니 GPT 구현 및 텍스트 생성

아래 코드를 순서대로 실행하여 GPT의 동작 원리를 직접 확인합니다.

### 5-1. Causal Mask 시각화


In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib


In [ ]:
import torch
import matplotlib.pyplot as plt

seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
tokens = ["나는", "영화를", "재미있게", "봤다", "[끝]"]

fig, ax = plt.subplots(figsize=(5, 4.5))
display = (~mask).float().numpy()
ax.imshow(display, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
ax.set_xlabel("참조 대상 (Key)")
ax.set_ylabel("현재 위치 (Query)")
ax.set_title("Causal Mask (초록=참조 가능, 빨강=가림)")

for i in range(seq_len):
    for j in range(seq_len):
        label = "O" if not mask[i][j] else "X"
        color = "black" if not mask[i][j] else "white"
        ax.text(j, i, label, ha="center", va="center", fontsize=14, color=color)

plt.tight_layout()
plt.show()
print("각 단어는 자기보다 왼쪽(과거)만 참조 가능 → 텍스트 생성의 핵심 구조")


### 5-2. 미니 GPT 구현 및 학습


In [ ]:
import torch
import torch.nn as nn
import math

# 학습 데이터
sentences = [
    "나는 영화를 좋아한다",
    "나는 음악을 좋아한다",
    "그는 영화를 싫어한다",
    "그는 음악을 싫어한다",
]

# 어휘 사전
all_tokens = sorted(set(w for s in sentences for w in s.split()))
word2idx = {w: i+1 for i, w in enumerate(all_tokens)}
word2idx["<PAD>"] = 0
idx2word = {i: w for w, i in word2idx.items()}
print(f"어휘 사전: {word2idx}")


In [ ]:
# "다음 단어 예측" 학습 쌍 만들기
MAX_LEN = 4
X_train = []
y_train = []

for sent in sentences:
    tokens = [word2idx[w] for w in sent.split()]
    for i in range(1, len(tokens)):
        inp = tokens[:i]
        padded = [0] * (MAX_LEN - len(inp)) + inp
        X_train.append(padded)
        y_train.append(tokens[i])

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

print(f"학습 샘플: {len(X_train)}개\n")
for i in range(min(6, len(X_train))):
    words_in = [idx2word[t.item()] for t in X_train[i] if t.item() != 0]
    word_out = idx2word[y_train[i].item()]
    print(f"  {str(words_in):30s} → \"{word_out}\"")


In [ ]:
# 미니 GPT 모델 정의
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.token_emb(x) + self.pos_emb(pos)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        x = self.transformer(x, mask=causal_mask)
        logits = self.fc_out(x[:, -1, :])
        return logits

mini_gpt = MiniGPT(vocab_size=len(word2idx), d_model=32, num_heads=4, num_layers=2, max_len=MAX_LEN)
print(mini_gpt)
print(f"\n파라미터: {sum(p.numel() for p in mini_gpt.parameters()):,}개")


In [ ]:
# 학습
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mini_gpt.parameters(), lr=0.005)

for epoch in range(200):
    mini_gpt.train()
    logits = mini_gpt(X_train)
    loss = criterion(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        acc = (logits.argmax(dim=1) == y_train).float().mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Acc: {acc.item():.0%}")

print("\n학습 완료!")


In [ ]:
# 텍스트 생성 (자기회귀 방식)
def generate(start_words, max_generate=3):
    mini_gpt.eval()
    tokens = [word2idx.get(w, 0) for w in start_words]
    result = list(start_words)
    for _ in range(max_generate):
        padded = [0] * (MAX_LEN - len(tokens)) + tokens[-MAX_LEN:]
        inp = torch.tensor([padded], dtype=torch.long)
        with torch.no_grad():
            logits = mini_gpt(inp)
            next_id = logits.argmax(dim=1).item()
        if next_id == 0:
            break
        tokens.append(next_id)
        result.append(idx2word[next_id])
    return " ".join(result)

print("=== 미니 GPT 텍스트 생성 ===")
print(f"  '나는'         → {generate(['나는'])}")
print(f"  '그는'         → {generate(['그는'])}")
print(f"  '나는 음악을'  → {generate(['나는', '음악을'])}")
print(f"  '그는 영화를'  → {generate(['그는', '영화를'])}")
print()
print("→ 학습한 패턴대로 다음 단어를 예측합니다.")
print("→ 이 원리를 수억 배 규모로 키운 것이 ChatGPT입니다.")


### 5-3. 한국어 GPT(KoGPT2)로 실제 텍스트 생성

SKT에서 한국어 텍스트로 학습한 `skt/kogpt2-base-v2` 모델을 사용합니다.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="skt/kogpt2-base-v2",
)

results = generator(
    "인공지능 기술이 발전하면",
    max_length=80,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.8,
)

print("=== 한국어 GPT 텍스트 생성 ===")
for i, r in enumerate(results, 1):
    print(f"\n[생성 {i}]")
    print(r["generated_text"])


In [ ]:
# Temperature 비교
prompt = "오늘 날씨가 좋아서"
print(f'프롬프트: "{prompt}"\n')

for temp in [0.3, 0.7, 1.2]:
    result = generator(
        prompt,
        max_length=60,
        do_sample=True,
        temperature=temp,
        num_return_sequences=1,
    )
    text = result[0]["generated_text"]
    print(f"Temperature={temp}:")
    print(f"  {text}")
    print()


In [ ]:
# 다양한 프롬프트 실험
prompts = [
    "대한민국의 수도는",
    "맛있는 음식을 먹으면",
    "프로그래밍을 배우려면",
    "행복한 삶을 위해서는",
]

print("=== 다양한 프롬프트 실험 ===\n")
for prompt in prompts:
    result = generator(
        prompt,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
    )
    print(f'입력: "{prompt}"')
    print(f'생성: {result[0]["generated_text"]}')
    print()


---
## 6. GPT 버전별 성능 비교와 최신 모델과의 비교점

### 6-1. GPT 버전별 발전

| 모델 | 출시 | 파라미터 수 | 핵심 특징 |
|------|------|-----------|----------|
| **GPT-1** | 2018.06 | 1.17억 | Transformer Decoder로 사전학습 + 파인튜닝 개념 제시 |
| **GPT-2** | 2019.02 | 15억 | 규모를 키우면 성능이 올라감을 입증. Zero-shot 가능성 제시 |
| **GPT-3** | 2020.05 | 1,750억 | Few-shot 프롬프트만으로 다양한 과제 수행. 파인튜닝 없이도 강력 |
| **GPT-3.5** | 2022.03 | 비공개 | RLHF(인간 피드백 강화학습) 적용. ChatGPT의 기반 |
| **GPT-4** | 2023.03 | 비공개 | 멀티모달(텍스트+이미지). 추론 능력 대폭 향상 |
| **GPT-4o** | 2024.05 | 비공개 | 텍스트/이미지/음성 통합 처리. 속도와 비용 개선 |

### 6-2. 발전의 핵심 패턴

GPT의 발전에서 관찰되는 핵심 패턴은 세 가지입니다.

**① 규모의 법칙 (Scaling Law)**
모델 크기, 데이터 양, 연산량을 키우면 성능이 예측 가능하게 향상됩니다.
GPT-1(1억)에서 GPT-3(1,750억)으로 파라미터를 1,500배 키우자, 단순한 텍스트 생성을 넘어 Few-shot 학습이 가능해졌습니다.

**② 창발적 능력 (Emergent Abilities)**
모델이 일정 규모를 넘으면 학습하지 않은 능력이 갑자기 나타납니다.
산술 연산, 논리적 추론, 코드 작성 등은 명시적으로 학습시키지 않았지만, 모델이 충분히 커지자 자연스럽게 가능해졌습니다.

**③ 정렬 (Alignment)**
GPT-3까지는 "다음 단어 예측"만 잘하는 모델이었지만, GPT-3.5부터 RLHF를 적용하여 사람의 의도에 맞게 답변하도록 조정했습니다.
이것이 ChatGPT가 단순한 텍스트 생성기가 아닌 "대화 파트너"가 될 수 있었던 핵심입니다.

### 6-3. 실습에서 사용한 모델(KoGPT2)과 최신 모델의 차이

| 비교 항목 | KoGPT2 (실습) | GPT-4 / Claude (최신) |
|-----------|--------------|----------------------|
| 파라미터 수 | 약 1.25억 | 수천억 이상 (추정) |
| 학습 데이터 | 한국어 텍스트 (제한적) | 다국어 + 코드 + 논문 등 |
| 프롬프트 성능 | Few-shot 제한적 | Zero/Few-shot 매우 우수 |
| 추론 능력 | 거의 없음 | 수학, 논리, 코딩 가능 |
| RLHF | 미적용 | 적용 (사람 의도에 정렬) |
| 멀티모달 | 텍스트만 | 텍스트 + 이미지 + 음성 |

실습에서 KoGPT2의 프롬프트 성능이 제한적이었던 이유는 모델 규모가 작기 때문입니다.
같은 Transformer 구조라도 규모를 키우고, RLHF로 정렬하면 ChatGPT 수준의 성능에 도달합니다.

### 6-4. 향후 학습할 내용과의 연결

```
[지금까지]                        [앞으로]
Transformer 구조 이해     →     RAG (검색 증강 생성)
GPT 사전학습 원리 이해    →     파인튜닝 (특정 도메인 적응)
프롬프트 설계 기초        →     LLM 애플리케이션 개발
```

| 개념 | 의미 | 필요한 이해 |
|------|------|------------|
| **파인튜닝** | 사전학습된 GPT의 가중치를 특정 데이터로 미세 조정 | Self-Attention 가중치가 어떻게 업데이트되는지 |
| **RAG** | 외부 문서를 검색하여 프롬프트에 추가 → 최신/전문 정보 반영 | 임베딩 벡터로 문서 유사도를 계산하는 원리 |
| **LLM 앱 개발** | GPT API + 프롬프트 엔지니어링으로 실제 서비스 구축 | Temperature, 토큰 제한 등 생성 파라미터 이해 |


---
## 전체 핵심 정리

### GPT 핵심 공식

```
GPT = Transformer + Causal Mask + 다음 단어 예측
```

### 텍스트 모델 발전 경로

```
BoW/TF-IDF → Word2Vec → RNN/LSTM → Transformer → GPT → ChatGPT
 단어 빈도    단어 의미   순서 이해   문맥 이해     생성    대화
```

### 세 가지 패러다임

```
패러다임 1: TF-IDF + ML         사람이 특성을 설계
패러다임 2: 사전학습 + 파인튜닝   모델이 언어를 학습
패러다임 3: 프롬프트              질문만으로 해결
```

### Transformer → GPT 차이점

```
Encoder (양방향) → 분류/이해 (BERT)
Decoder (단방향) → 텍스트 생성 (GPT)

차이: Causal Mask 한 줄 + 출력층 변경
```

### GPT 발전 핵심

```
규모↑ → 성능↑ → 창발적 능력 → RLHF 정렬 → ChatGPT
```
